<center>
<img src="../../img/ods_stickers.jpg">
    
## [mlcourse.ai](https://mlcourse.ai) - دورة التعلم الآلي المفتوحة
المؤلف: فيتالي رادشينكو. يتم توزيع كل المحتوى بموجب ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/).



# <center>المهمة رقم 5 (تجريبي)</center>
## <center>الانحدار اللوجستي والغابة العشوائية في مشكلة تسجيل الائتمان</center> 
** نفس المهمة مثل [Kaggle Kernel](https://www.kaggle.com/kashnitsky/a5-demo-logit-and-rf-for-credit-scoring) + [الحل](https://www.kaggle.com/kashnitsky/a5-demo-logit-and-rf-for-credit-scoring-sol).**



في هذا التدريب، ستقوم ببناء نماذج والإجابة على الأسئلة باستخدام البيانات المتعلقة بالدرجات الائتمانية.
الرجاء كتابة الرمز الخاص بك في الخلايا التي تحتوي على العنصر النائب "الرمز الخاص بك هنا". ثم أجب عن الأسئلة في [النموذج](https://docs.google.com/forms/d/1gKt0DA4So8ohKAHZNCk58ezvg7K_tik26d9QND7WC6M/edit).
لنبدأ بتمرين الإحماء.



**السؤال 1.** يوجد 5 محلفين في قاعة المحكمة. يمكن لكل واحد منهم تحديد ذنب المدعى عليه بشكل صحيح بنسبة 70٪، بشكل مستقل عن بعضها البعض. ما هو احتمال أن يتوصل المحلفون معًا إلى الحكم الصحيح إذا كان القرار النهائي بأغلبية الأصوات؟
1.70.00%
2.83.20%
3.83.70%
4.87.50%



عظيم! دعنا ننتقل إلى التعلم الآلي.
## إعداد مشكلة تسجيل الائتمان
#### مشكلة
توقع ما إذا كان العميل سيسدد رصيده خلال 90 يومًا. هذه مشكلة تصنيف ثنائي؛ سنقوم بتصنيف العملاء إلى فئات جيدة أو سيئة بناءً على توقعاتنا.
#### وصف البيانات| ميزة | نوع متغير | نوع القيمة | الوصف |
|:--------|:-------------|:----------|:-----------|
| العمر | ميزة الإدخال | عدد صحيح | عمر العميل |
| نسبة الديون | ميزة الإدخال | حقيقي | إجمالي دفعات القروض الشهرية (قرض، نفقة، الخ) / نسبة إجمالي الدخل الشهري |
| NumberOfTime30-59DaysPastDueNotWorse | ميزة الإدخال | عدد صحيح | عدد الحالات التي تأخر فيها العميل عن سداد قروض أخرى لمدة 30-59 يومًا (وليس أسوأ) خلال العامين الماضيين |
| NumberOfTimes90DaysLate | ميزة الإدخال | عدد صحيح | عدد الحالات التي تأخر فيها العميل عن سداد 90+ يوم على الاعتمادات الأخرى |
| NumberOfTime60-89DaysPastDueNotWorse | ميزة الإدخال | عدد صحيح | عدد الحالات عندما يكون لدى العميل 60-89dpd (ليس أسوأ) خلال العامين الماضيين |
| عدد المعتمدين | ميزة الإدخال | عدد صحيح | عدد المعالين من العملاء |
| خطيرةDlqin2yrs | المتغير المستهدف | ثنائي: <br>0 أو 1 | لم يقم العميل بسداد دين القرض خلال 90 يومًا |



فلنجهز بيئتنا:


In [ ]:
# Disable warnings in Anaconda
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()

In [ ]:
from matplotlib import rcParams

rcParams["figure.figsize"] = 11, 8


لنكتب الدالة التي ستستبدل قيم *NaN* بالوسيط لكل عمود.


In [ ]:
def fill_nan(table):
    for col in table.columns:
        table[col] = table[col].fillna(table[col].median())
    return table


والآن إقرأ البيانات:


In [ ]:
data = pd.read_csv("../../data/credit_scoring_sample.csv", sep=";")
data.head()


انظر إلى أنواع المتغيرات:


In [ ]:
data.dtypes


التحقق من توازن الفصل:


In [ ]:
ax = data["SeriousDlqin2yrs"].hist(orientation="horizontal", color="red")
ax.set_xlabel("number_of_observations")
ax.set_ylabel("unique_value")
ax.set_title("Target distribution")

print("Distribution of the target:")
data["SeriousDlqin2yrs"].value_counts() / data.shape[0]


افصل أسماء متغيرات الإدخال عن طريق استبعاد الهدف:


In [ ]:
independent_columns_names = [x for x in data if x != "SeriousDlqin2yrs"]
independent_columns_names


قم بتطبيق الوظيفة لاستبدال قيم *NaN*:


In [ ]:
table = fill_nan(data)


افصل بين المتغير المستهدف وميزات الإدخال:


In [ ]:
X = table[independent_columns_names]
y = table["SeriousDlqin2yrs"]


## التمهيد



**السؤال 2.** قم بإجراء تقدير فاصل لمتوسط عمر العملاء الذين أخروا السداد عند مستوى ثقة 90%. استخدم المثال من المقالة كمرجع، إذا لزم الأمر. استخدم أيضًا `np.random.seed(0)` كما كان من قبل. ما هو تقدير الفاصل الزمني الناتج؟
1.52.59 - 52.86
2.45.71 - 46.13
3.45.68 - 46.17
4.52.56 - 52.88


In [ ]:
# Your code here

## الانحدار اللوجستي



دعونا نستعد لاستخدام الانحدار اللوجستي:


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold


الآن، سنقوم بإنشاء نموذج `LogisticRegression` واستخدام `class_weight='balanced'` لتعويض فصولنا غير المتوازنة.


In [ ]:
lr = LogisticRegression(random_state=5, class_weight="balanced")


دعونا نحاول العثور على أفضل معامل تنظيم، وهو معامل `C` للانحدار اللوجستي. بعد ذلك، سيكون لدينا نموذج أمثل غير مفرط وهو مؤشر جيد للمتغير المستهدف.


In [ ]:
parameters = {"C": (0.0001, 0.001, 0.01, 0.1, 1, 10)}


من أجل العثور على القيمة المثلى لـ `C`، دعنا نطبق التحقق الطبقي 5 أضعاف وننظر إلى *ROC AUC* مقابل قيم مختلفة للمعلمة `C`. استخدم الدالة `StratifiedKFold` لهذا: 


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=5)


أحد المقاييس المهمة لجودة النموذج هو *المنطقة أسفل المنحنى (AUC)*. *ROC AUC* يختلف من 0 إلى 1. كلما اقتربت ROC AUC من 1، كانت جودة نموذج التصنيف أفضل.



**السؤال 3.** قم بإجراء *بحث شبكي* باستخدام مقياس النقاط "roc_auc" للمعلمة `C`. ما هي قيمة المعلمة `C` الأمثل؟ 
1.0.0001
2.0.001
3.0.01
4.0.1
5. 1
6. 10


In [ ]:
# Your code here


**السؤال 4.** هل يمكننا اعتبار النموذج الأفضل مستقرًا؟ يكون النموذج *مستقرًا* إذا كان الانحراف المعياري عند التحقق أقل من 0.5%. احفظ قيمة *ROC AUC* لأفضل نموذج؛ سيكون مفيدًا للمهام التالية.
1. نعم
2. لا


In [ ]:
# Your code here


## أهمية الميزة
**السؤال 5.** *يتم تحديد أهمية الميزة* من خلال القيمة المطلقة للمعامل المقابل لها. أولاً، تحتاج إلى تسوية جميع قيم الميزات بحيث تكون صالحة للمقارنة بينها. ما هي الميزة الأكثر أهمية لأفضل نموذج الانحدار اللوجستي؟1. العمر
2. NumberOfTime30-59DaysPastDueليس أسوأ
3. نسبة الديون
4. عدد الأوقات 90 يومًا متأخرًا
5. NumberOfTime60-89DaysPastDueليس أسوأ
6. الدخل الشهري
7. عدد المعالين


In [ ]:
# Your code here


**السؤال 6.** احسب مدى تأثير `DebtRatio` على تنبؤاتنا باستخدام [وظيفة softmax](https://en.wikipedia.org/wiki/Softmax_function). ما هي قيمته؟
1.0.38
2.-0.02
3.0.11
4.0.24


In [ ]:
# Your code here


**السؤال 7.** دعونا نرى كيف يمكننا تفسير تأثير ميزاتنا. لهذا، قم بإعادة حساب الانحدار اللوجستي بالقيم المطلقة، أي بدون قياس. بعد ذلك، تعديل عمر العميل بإضافة 20 سنة، مع إبقاء الميزات الأخرى دون تغيير. كم مرة ستزداد احتمالية عدم سداد العميل لديونه؟ يمكنك العثور على مثال للحساب النظري [هنا](https://www.unm.edu/~schrader/biostat/bio2/Spr06/lec11.pdf).
1.-0.01
2.0.70
3.8.32
4.0.66


In [ ]:
# Your code here


## غابة عشوائية



استيراد مصنف الغابة العشوائية:


In [ ]:
from sklearn.ensemble import RandomForestClassifier


تهيئة الغابة العشوائية بـ 100 شجرة وموازنة الفئات المستهدفة:


In [ ]:
rf = RandomForestClassifier(
    n_estimators=100, n_jobs=-1, random_state=42, class_weight="balanced"
)


سنبحث عن أفضل المعلمات من بين القيم التالية:


In [ ]:
parameters = {
    "max_features": [1, 2, 4],
    "min_samples_leaf": [3, 5, 7, 9],
    "max_depth": [5, 10, 15],
}


أيضًا، سوف نستخدم التحقق الطبقي k-fold مرة أخرى. يجب أن يظل لديك المتغير `skf`.



**السؤال 8.** ما مدى ارتفاع *ROC AUC* لأفضل نموذج غابة عشوائي عن أفضل انحدار لوجستي عند التحقق من الصحة؟
1.4%
2.3%
3.2%
4.1%


In [ ]:
# Your code here


**السؤال 9.** ما الميزة ذات التأثير الأضعف في نموذج Random Forest؟
1. العمر
2. NumberOfTime30-59DaysPastDueليس أسوأ
3. نسبة الديون
4. عدد الأوقات 90 يومًا متأخرًا
5. NumberOfTime60-89DaysPastDueليس أسوأ
6. الدخل الشهري
7. عدد المعالين


In [ ]:
# Your code here


**السؤال 10.** ما هي أهم ميزة لاستخدام *الانحدار اللوجستي* مقابل *المجموعة العشوائية* لهذه المشكلة؟1. قضاء وقت أقل في تركيب النموذج؛
2. متغيرات أقل للتكرار؛
3. ميزة التفسير.
4. الخصائص الخطية للخوارزمية.



## التعبئة



وحدات الاستيراد وإعداد المعلمات للتعبئة:


In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_val_score

parameters = {
    "max_features": [2, 3, 4],
    "max_samples": [0.5, 0.7, 0.9],
    "base_estimator__C": [0.0001, 0.001, 0.01, 1, 10, 100],
}


**السؤال 11.** قم بتركيب مصنف التعبئة باستخدام `random_state=42`. بالنسبة للمصنفات الأساسية، استخدم 100 تراجع لوجستي واستخدم `RandomizedSearchCV` بدلاً من `GridSearchCV`. سوف يستغرق الأمر الكثير من الوقت للتكرار على جميع المتغيرات الـ 54، لذا قم بتعيين الحد الأقصى لعدد التكرارات لـ `RandomizedSearchCV` إلى 20. لا تنس تعيين المعلمات `cv` و`random_state=1`. ما هو أفضل *ROC AUC* الذي حققته؟
1. 80.75%
2. 80.12%
3.79.62%
4.76.50%


In [ ]:
# Your code here


**السؤال 12.** أعط تفسيرًا لأفضل المعلمات للتعبئة. لماذا تعتبر قيم `max_features` و`max_samples` هي الأفضل؟
1. بالنسبة للتعبئة، من المهم استخدام أقل عدد ممكن من الميزات؛
2. التعبئة تعمل بشكل أفضل على العينات الصغيرة؛
3. ارتباط أقل بين النماذج الفردية؛
4. كلما زاد عدد الميزات، قل فقدان المعلومات.